# Portfolio v6: exact-metric chemistry offset refit

This experimental revision starts from the fully verified 10.88048 portfolio.
It keeps the Q82 inverse-square-root and Q898 DQI circuits byte-identical and
tries one safe numerical improvement: refit the R26 factorization's `H2_I`
offsets while keeping every decoded angle word and every chemistry circuit
fixed.

The optimizer uses an analytic gradient for `lambda_eff`, searches several
bounded trust regions, and evaluates every finalist with the authoritative
`evaluate_words` and `passes_accuracy` functions. A candidate is selected only
when it improves `lambda_eff` and still satisfies both 20 mHa limits, the strict
Lambda/E_gap floors and the valid spectrum-amplification domain. Otherwise the
notebook retains the exact v5 chemistry NPZ.

After selection it repeats the complete v5 review: all 200,540,928 invsqrt
shots, four independent 102,400-shot DQI trials, all 104,704 chemistry shots,
the extra metric-format checks and ZIP CRC validation.

This refit tests the remaining safe route toward a modest score-11-class
improvement. Reaching the 808k DQI or inner-QROAM-cliff targets still requires
circuit/trainer source that is not present in the public artifact datasets.


In [ ]:
import hashlib
import json
import math
import os
import pathlib
import platform
import shutil
import sys
import threading
import time
import zipfile

os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
)

import numpy as np
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get("QCC_INPUT", "/kaggle/input"))
WORK = pathlib.Path(os.environ.get("QCC_WORK", "/kaggle/working"))
SUB = pathlib.Path(os.environ.get("QCC_SUB", "/tmp/qcc_submission_v5"))
SLUG = "google-verifiable-quantum-circuit-compilation"
REFERENCE = {
    "invsqrt": 2076774.233715282,
    "dqi": 24439474177519.598,
    "dfthcsos": 3355599588.3155365,
}
PLAN = {
    "invsqrt": {
        "dataset_version_id": 21216223,
        "slug": "qcc-best-invsqrt-circuit",
        "rel": "invsqrt.kmb",
        "sha256": "0f84f5bab3a02e6b357d4656553607174345a33d252cef643f9136db13212141",
        "reported_cost": 138095.13529214897,
    },
    "dqi": {
        "dataset_version_id": 21211427,
        "slug": "qcc-best-dqi",
        "rel": "dqi.kmb",
        "sha256": "c5da9cd4e2d73dd41417e539f2659009d646b3ee30ddb5ca494261378a502856",
        "reported_cost": 1045962255881.3461,
    },
    "dfthcsos": {
        "dataset_version_id": 21216621,
        "slug": "qcc-best-circuits",
        "rel": ".",
        "sha256": "47187a5d3664c76e48f1266c39a24b55a7bcc845743d6a673c26c8e69b5cbcee",
        "reported_cost": 919608140.5572633,
    },
}


def sha256_file(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def content_sha256(path):
    """Match the portfolio study's published file/folder digest exactly."""
    path = pathlib.Path(path)
    digest = hashlib.sha256()
    if path.is_dir():
        for child in sorted(item for item in path.rglob("*") if item.is_file()):
            digest.update(child.relative_to(path).as_posix().encode())
            digest.update(child.read_bytes())
    else:
        digest.update(path.read_bytes())
    return digest.hexdigest()


def competition_candidates(root):
    preferred = [root, root / "competitions" / SLUG, root / SLUG]
    discovered = [path.parent for path in root.rglob("kickmix") if path.is_dir()]
    unique = []
    for path in preferred + sorted(discovered):
        path = path.resolve()
        if path not in unique and (path / "kickmix").is_dir() and (path / "verifiers").is_dir():
            unique.append(path)
    return unique


def unique_dataset_root(slug):
    hits = sorted(
        path.resolve() for path in INPUT.rglob(slug)
        if path.is_dir() and path.name == slug
    )
    unique = list(dict.fromkeys(hits))
    if len(unique) != 1:
        raise RuntimeError(f"Expected exactly one {slug!r} dataset directory; found {unique}")
    return unique[0]


if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f"kickmix.so requires CPython 3.13; found {platform.python_version()}")
if not INPUT.exists():
    raise FileNotFoundError(INPUT)
competition_dirs = competition_candidates(INPUT)
if len(competition_dirs) != 1:
    raise RuntimeError(f"Expected one competition input; found {competition_dirs}")
COMP = competition_dirs[0]

SOURCES = {}
for task, entry in PLAN.items():
    root = unique_dataset_root(entry["slug"])
    source = (root / entry["rel"]).resolve()
    if not source.exists():
        raise FileNotFoundError(f"{task}: expected {source}")
    observed = content_sha256(source)
    if observed != entry["sha256"]:
        raise AssertionError(
            f"{task}: dataset bytes changed; expected {entry['sha256']}, got {observed}"
        )
    SOURCES[task] = source
    print(f"{task:9s} dataset v{entry['dataset_version_id']} hash OK: {observed}")

WORK.mkdir(parents=True, exist_ok=True)
shutil.rmtree(SUB, ignore_errors=True)
(SUB / "dfthcsos").mkdir(parents=True)
shutil.copy2(SOURCES["invsqrt"], SUB / "invsqrt.kmb")
shutil.copy2(SOURCES["dqi"], SUB / "dqi.kmb")

chem_source = SOURCES["dfthcsos"]
submission_files = sorted(chem_source.glob("submission.npz"))
angle_files = sorted(
    chem_source.glob("angle_k_*.kmb"),
    key=lambda path: int(path.stem.rsplit("_", 1)[1]),
)
if len(submission_files) != 1:
    raise RuntimeError(f"Expected one chemistry submission.npz; found {submission_files}")
indices = [int(path.stem.rsplit("_", 1)[1]) for path in angle_files]
if indices != list(range(len(indices))):
    raise RuntimeError(f"Chemistry angle circuit indices are not contiguous: {indices}")
for source in [*angle_files, submission_files[0]]:
    shutil.copy2(source, SUB / "dfthcsos" / source.name)
if len(angle_files) != 25:
    raise RuntimeError(f"Expected 25 R26 angle circuits, found {len(angle_files)}")

sys.path[:0] = [str(COMP / "kickmix"), str(COMP / "verifiers")]
import kickmix as km

available_cpus = os.cpu_count() or 4
WORKERS = max(1, min(int(os.environ.get("QCC_WORKERS", available_cpus)), available_cpus, 4))
print(f"Competition data: {COMP}")
print(f"Submission tree : {SUB}")
print(f"Workers / CPUs  : {WORKERS} / {available_cpus}")


In [ ]:
# Embed and test the reviewed fixed-word offset optimizer.
REFIT_DIR = WORK / "v6_refit_source"
REFIT_DIR.mkdir(parents=True, exist_ok=True)
(REFIT_DIR / "dfthcsos_offset_refit.py").write_text('#!/usr/bin/env python3\n"""Verifier-aligned fixed-word offset refit for a DFTHC-SOS submission.\n\nThe ANGLE circuits and all ``W`` tensor entries stay fixed.  Only the stored\n``H2_I`` array is changed, which changes the verifier\'s offset\n``W_offset = H2_I - sum_b(W)``.  Candidate selection always uses the official\n``evaluate_words`` and ``passes_accuracy`` functions.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, replace\nimport math\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nimport numpy as np\nfrom scipy.optimize import minimize\n\n\n@dataclass\nclass OffsetModel:\n    """Fast lambda objective and analytic gradient at fixed decoded vectors."""\n\n    h0: np.ndarray\n    x: np.ndarray\n    w: np.ndarray\n    beta_1: float\n    ecore: float\n    electrons: int\n    energy_upper: float\n    start: np.ndarray\n\n    def __post_init__(self) -> None:\n        self.x = np.asarray(self.x, dtype=np.float64)\n        self.w = np.asarray(self.w, dtype=np.float64)\n        self.start = np.asarray(self.start, dtype=np.float64)\n        self.sum_w = np.sum(self.w, axis=1)\n        self.l1_w = np.linalg.norm(self.w, ord=1, axis=1)\n        self.trace_x = np.trace(self.x, axis1=-2, axis2=-1)\n\n    def value_gradient(\n        self,\n        flat: np.ndarray,\n        *,\n        gap_floor: float = 3.914899,\n        gap_margin: float = 2e-6,\n        floor_penalty: float = 2e6,\n        trust_penalty: float = 1e-7,\n    ) -> tuple[float, np.ndarray]:\n        offsets = np.asarray(flat, dtype=np.float64).reshape(self.start.shape)\n        h1_prime = self.h0 - np.einsum("rc,rcpq->pq", offsets, self.x, optimize=True)\n        eigenvalues, eigenvectors = np.linalg.eigh(h1_prime)\n        signs = np.sign(eigenvalues)\n        sign_h = (eigenvectors * signs[None, :]) @ eigenvectors.T\n        overlap = np.einsum("pq,rcpq->rc", sign_h, self.x, optimize=True)\n\n        lambda_h1 = float(np.sum(np.abs(eigenvalues)))\n        w_b = offsets + self.sum_w\n        widths = self.l1_w + np.abs(w_b)\n        lambda_h2 = 0.25 * float(np.sum(widths * widths))\n        Lambda = lambda_h1 + lambda_h2\n\n        trace_h1 = float(np.sum(eigenvalues))\n        e_sos = (\n            self.ecore\n            - self.electrons * self.beta_1\n            - (lambda_h1 - trace_h1)\n            - 0.5 * float(np.sum(offsets * offsets))\n        )\n        e_gap = self.energy_upper - e_sos\n        radicand = e_gap * (2.0 * Lambda - e_gap)\n        if not np.isfinite(radicand) or radicand <= 0:\n            # A large smooth-ish rejection value keeps L-BFGS-B inside domain.\n            delta = offsets - self.start\n            return 1e9 + float(np.sum(delta * delta)), (2.0 * delta).ravel()\n\n        lambda_eff = math.sqrt(radicand)\n        grad_lambda_h1 = -overlap\n        grad_lambda_h2 = 0.5 * widths * np.sign(w_b)\n        grad_Lambda = grad_lambda_h1 + grad_lambda_h2\n        grad_gap = -overlap + self.trace_x + offsets\n        gradient = (\n            (Lambda - e_gap) * grad_gap + e_gap * grad_Lambda\n        ) / lambda_eff\n\n        objective = lambda_eff\n        minimum_gap = gap_floor + gap_margin\n        if e_gap < minimum_gap:\n            shortfall = minimum_gap - e_gap\n            objective += floor_penalty * shortfall * shortfall\n            gradient += -2.0 * floor_penalty * shortfall * grad_gap\n        if Lambda <= 49.000001:\n            shortfall = 49.000001 - Lambda\n            objective += floor_penalty * shortfall * shortfall\n            gradient += -2.0 * floor_penalty * shortfall * grad_Lambda\n\n        delta = offsets - self.start\n        objective += trust_penalty * float(np.sum(delta * delta))\n        gradient += 2.0 * trust_penalty * delta\n        return float(objective), gradient.ravel()\n\n\ndef _build_model(inputs: Any, words: np.ndarray, edges: np.ndarray, brot: int) -> OffsetModel:\n    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors, reconstruct_x, split_u_vectors\n\n    vectors = reconstruct_u_vectors(words / (1 << (brot - 1)), edges)\n    norms = np.linalg.norm(vectors, axis=1, keepdims=True)\n    one_body, two_body = split_u_vectors(vectors / norms, inputs.w)\n    del one_body\n    x = reconstruct_x(two_body, inputs.w)\n    x = np.tril(x) + np.swapaxes(np.tril(x, k=-1), -1, -2)\n    tei = np.asarray(inputs.ref["tei"], dtype=np.float64)\n    oei = np.asarray(inputs.ref["oei"], dtype=np.float64)\n    h1_thc = oei - 0.5 * np.einsum("prrq->pq", tei)\n    h0 = (\n        h1_thc\n        - 0.5 * inputs.ref["electrons"] * inputs.S\n        + inputs.beta_1 * np.eye(inputs.num_orbitals)\n    )\n    return OffsetModel(\n        h0=h0,\n        x=x,\n        w=inputs.w,\n        beta_1=inputs.beta_1,\n        ecore=float(inputs.ref["ecore"]),\n        electrons=int(inputs.ref["electrons"]),\n        energy_upper=float(inputs.energy_upper),\n        start=inputs.W,\n    )\n\n\ndef _serializable(metrics: dict[str, Any]) -> dict[str, Any]:\n    return {\n        key: value.item() if isinstance(value, np.generic) else value\n        for key, value in metrics.items()\n    }\n\n\ndef refit_offsets(\n    factorization: dict[str, np.ndarray],\n    words: np.ndarray,\n    edges: np.ndarray,\n    brot: int,\n    *,\n    radii: tuple[float, ...] = (2e-4, 6e-4, 2e-3, 6e-3, 2e-2),\n    maxiter: int = 80,\n) -> tuple[dict[str, np.ndarray], dict[str, Any]]:\n    """Return the best officially accepted H2_I-only refit and its report."""\n    from dfthcsos.verify import evaluate_words, load_inputs, passes_accuracy\n\n    original = {name: np.array(value) for name, value in factorization.items()}\n    inputs = load_inputs(original)\n    baseline_metrics = evaluate_words(inputs, words, edges, brot)\n    if not passes_accuracy(baseline_metrics):\n        raise ValueError(f"baseline factorization is not accepted: {baseline_metrics}")\n\n    model = _build_model(inputs, words, edges, brot)\n    start = np.array(inputs.W, dtype=np.float64)\n    best_params = original\n    best_metrics = baseline_metrics\n    trials: list[dict[str, Any]] = []\n\n    for radius in radii:\n        widths = radius * (1.0 + np.abs(start))\n        bounds = list(zip((start - widths).ravel(), (start + widths).ravel()))\n\n        def function(flat: np.ndarray) -> tuple[float, np.ndarray]:\n            return model.value_gradient(flat)\n\n        result = minimize(\n            function,\n            start.ravel(),\n            method="L-BFGS-B",\n            jac=True,\n            bounds=bounds,\n            options={"maxiter": maxiter, "ftol": 1e-13, "gtol": 1e-9, "maxls": 30},\n        )\n        offsets = np.asarray(result.x).reshape(start.shape)\n        candidate = {name: np.array(value) for name, value in original.items()}\n        candidate["H2_I"] = offsets + np.sum(inputs.w, axis=1)\n        candidate_inputs = replace(inputs, params=candidate, W=offsets)\n        metrics = evaluate_words(candidate_inputs, words, edges, brot)\n        accepted = bool(passes_accuracy(metrics))\n        improved = accepted and float(metrics["lambda_eff"]) < float(best_metrics["lambda_eff"])\n        trials.append({\n            "radius": radius,\n            "optimizer_success": bool(result.success),\n            "optimizer_status": int(result.status),\n            "optimizer_message": str(result.message),\n            "iterations": int(result.nit),\n            "evaluations": int(result.nfev),\n            "accepted": accepted,\n            "improved": improved,\n            "max_offset_change": float(np.max(np.abs(offsets - start))),\n            "metrics": _serializable(metrics),\n        })\n        if improved:\n            best_params = candidate\n            best_metrics = metrics\n\n    report = {\n        "method": "fixed decoded words; H2_I/offset refit only; official metric selection",\n        "baseline_metrics": _serializable(baseline_metrics),\n        "selected_metrics": _serializable(best_metrics),\n        "improved": float(best_metrics["lambda_eff"]) < float(baseline_metrics["lambda_eff"]),\n        "trials": trials,\n    }\n    return best_params, report\n\n\ndef gradient_check(seed: int = 20261025) -> float:\n    """Finite-difference check of the analytic objective gradient on a small model."""\n    rng = np.random.default_rng(seed)\n    r, c, n, b = 2, 3, 5, 4\n    x = rng.normal(scale=0.05, size=(r, c, n, n))\n    x = 0.5 * (x + np.swapaxes(x, -1, -2))\n    model = OffsetModel(\n        h0=np.diag(np.linspace(-2.0, 2.0, n)) + rng.normal(scale=0.01, size=(n, n)),\n        x=x,\n        w=rng.normal(scale=0.03, size=(r, b, c)),\n        beta_1=0.0,\n        ecore=-10.0,\n        electrons=4,\n        energy_upper=-7.0,\n        start=rng.normal(scale=0.05, size=(r, c)),\n    )\n    model.h0 = 0.5 * (model.h0 + model.h0.T)\n    point = model.start.ravel() + rng.normal(scale=1e-3, size=r * c)\n    _, analytic = model.value_gradient(\n        point, gap_floor=-1e6, floor_penalty=0.0, trust_penalty=0.0\n    )\n    numeric = np.zeros_like(point)\n    step = 1e-6\n    for index in range(len(point)):\n        plus, minus = point.copy(), point.copy()\n        plus[index] += step\n        minus[index] -= step\n        fp = model.value_gradient(plus, gap_floor=-1e6, floor_penalty=0.0, trust_penalty=0.0)[0]\n        fm = model.value_gradient(minus, gap_floor=-1e6, floor_penalty=0.0, trust_penalty=0.0)[0]\n        numeric[index] = (fp - fm) / (2 * step)\n    return float(np.max(np.abs(analytic - numeric)))\n\n\nif __name__ == "__main__":\n    error = gradient_check()\n    print(f"maximum gradient error: {error:.3e}")\n    if error > 2e-5:\n        raise SystemExit(1)\n')
sys.path.insert(0, str(REFIT_DIR))
import dfthcsos_offset_refit as OR

gradient_error = OR.gradient_check()
assert gradient_error < 2e-5, gradient_error
print(f"Analytic-gradient check: {gradient_error:.3e}")

from dfthcsos import verify as VC


def load_chemistry_circuit(path):
    return km.Circuit.from_file(str(path), format="kmb")


chemistry_paths = sorted(
    (SUB / "dfthcsos").glob("angle_k_*.kmb"),
    key=lambda path: int(path.stem.rsplit("_", 1)[1]),
)
chemistry_circuits = [load_chemistry_circuit(path) for path in chemistry_paths]
with np.load(SUB / "dfthcsos" / "submission.npz", allow_pickle=False) as archive:
    chemistry_archive = {name: np.array(archive[name]) for name in archive.files}
tree_edges_for_refit = np.array(chemistry_archive["tree_edges"])

print("Recovering the fixed decoded words with one shot per address", flush=True)
baseline_inputs = VC.load_inputs(chemistry_archive)
quick_evaluation = VC.evaluate_circuits(
    baseline_inputs,
    chemistry_circuits,
    tree_edges_for_refit,
    shots_per_address=1,
    seed=20261025,
)
angle_words_for_refit = np.array(quick_evaluation["angle_words"])
brot_for_refit = int(quick_evaluation["brot"])

print("Searching bounded H2_I/offset refits", flush=True)
selected_factorization, OFFSET_REFIT_REPORT = OR.refit_offsets(
    chemistry_archive,
    angle_words_for_refit,
    tree_edges_for_refit,
    brot_for_refit,
)
baseline_lambda = float(OFFSET_REFIT_REPORT["baseline_metrics"]["lambda_eff"])
selected_lambda = float(OFFSET_REFIT_REPORT["selected_metrics"]["lambda_eff"])

if OFFSET_REFIT_REPORT["improved"]:
    payload = {name: np.array(value) for name, value in selected_factorization.items()}
    payload["tree_edges"] = tree_edges_for_refit
    payload["angle_words"] = angle_words_for_refit
    temporary_npz = SUB / "dfthcsos" / "submission.refit.npz"
    np.savez(temporary_npz, **payload)
    temporary_npz.replace(SUB / "dfthcsos" / "submission.npz")
    print(f"Selected lambda_eff {baseline_lambda:.9f} -> {selected_lambda:.9f}", flush=True)
else:
    print(f"No accepted improvement below lambda_eff={baseline_lambda:.9f}; retaining v5", flush=True)

for trial in OFFSET_REFIT_REPORT["trials"]:
    metrics = trial["metrics"]
    print(
        f"radius={trial['radius']:.1e} accepted={trial['accepted']} "
        f"lambda={metrics['lambda_eff']:.9f} "
        f"e1={1e3 * metrics['error_1']:.4f}mHa "
        f"e2={1e3 * metrics['error_2']:.4f}mHa",
        flush=True,
    )
(WORK / "offset_refit_v6.json").write_text(
    json.dumps(OFFSET_REFIT_REPORT, indent=2, default=str) + "\n"
)
del baseline_inputs, quick_evaluation, selected_factorization
del chemistry_archive, chemistry_circuits, angle_words_for_refit
import gc
gc.collect()


In [ ]:
def load_circuit(path):
    return km.Circuit.from_file(str(path), format="kmb")


def with_heartbeat(label, function, every=60):
    finished = threading.Event()
    started = time.monotonic()

    def heartbeat():
        while not finished.wait(every):
            print(f"    {label} still running ({(time.monotonic() - started) / 60:.1f} min)", flush=True)

    monitor = threading.Thread(target=heartbeat, daemon=True)
    monitor.start()
    try:
        return function()
    finally:
        finished.set()
        monitor.join()


started = time.monotonic()
print("[1/3] Full inverse-square-root verification", flush=True)
from invsqrt import verify_submission_invsqrt as VI

invsqrt_result = with_heartbeat(
    "inverse-square-root verifier",
    lambda: VI.verify(
        load_circuit(SUB / "invsqrt.kmb"),
        VI.ProblemSpecInvSqrt.for_n_p(10),
        max_workers=WORKERS,
    ),
)
assert invsqrt_result["status"] == "PASS" and invsqrt_result["verified_shots"] == 200540928
assert invsqrt_result["num_qubits"] == 82, invsqrt_result

print("[2/3] Full DQI verification", flush=True)
from dqi import verify_submission_dqi as VD

dqi_circuit = load_circuit(SUB / "dqi.kmb")
dqi_trials = []
for salt in ("", "independent-v5-a", "independent-v5-b", "independent-v5-c"):
    result = with_heartbeat(
        f"DQI verifier salt={salt or '<empty>'}",
        lambda salt=salt: VD.verify(
            dqi_circuit,
            VD.ProblemSpecDQI.benchmark_instance(),
            salt=salt,
            max_workers=WORKERS,
        ),
    )
    assert result["status"] == "PASS" and result["verified_shots"] == 102400
    assert result["num_qubits"] == 898, result
    dqi_trials.append({"salt": salt, **result})
    print(
        f"    salt={salt or '<empty>'}: value={result['value_errors']}, "
        f"phase={result['phase_errors']}, R={result['repetition_factor']:.6f}, "
        f"cost={result['score']:.7g}",
        flush=True,
    )
dqi_result = dqi_trials[0]
dqi_worst_result = max(dqi_trials, key=lambda result: result["score"])

print("[3/3] Full DFTHC-SOS verification", flush=True)
from dfthcsos import verify as VC

circuits = [load_circuit(path) for path in sorted(
    (SUB / "dfthcsos").glob("angle_k_*.kmb"),
    key=lambda path: int(path.stem.rsplit("_", 1)[1]),
)]
with np.load(SUB / "dfthcsos" / "submission.npz", allow_pickle=False) as archive:
    factorization = {name: np.array(archive[name]) for name in archive.files}
tree_edges = factorization.pop("tree_edges")
dfthcsos_result = with_heartbeat(
    "chemistry verifier",
    lambda: VC.verify(circuits, factorization, tree_edges),
)
assert dfthcsos_result["status"] == "PASS" and dfthcsos_result["verified_shots"] == 104704
assert dfthcsos_result["num_qubits"] == 656, dfthcsos_result
assert dfthcsos_result["toffoli_count"] == 29360, dfthcsos_result

RESULTS = {
    "invsqrt": invsqrt_result,
    "dqi": dqi_result,
    "dfthcsos": dfthcsos_result,
}
SCORE = math.prod(REFERENCE[name] / RESULTS[name]["score"] for name in REFERENCE) ** (1 / 3)
WORST_SALT_SCORE = (
    (REFERENCE["invsqrt"] / invsqrt_result["score"])
    * (REFERENCE["dqi"] / dqi_worst_result["score"])
    * (REFERENCE["dfthcsos"] / dfthcsos_result["score"])
) ** (1 / 3)
if min(SCORE, WORST_SALT_SCORE) < 10.8:
    raise AssertionError(
        f"Portfolio fell below the v6 safety floor 10.8: default={SCORE:.6f}, "
        f"worst reviewed salt={WORST_SALT_SCORE:.6f}"
    )

rows = []
for name, label in (("invsqrt", "Inverse square root"), ("dqi", "DQI decoder"),
                    ("dfthcsos", "FeMoCo lookups")):
    result = RESULTS[name]
    rows.append({
        "Task": label,
        "Qubits": result["num_qubits"],
        "Toffoli": f"{result['toffoli_count']:,.3f}",
        "Cost": f"{result['score']:.7g}",
        "Reported cost": f"{PLAN[name]['reported_cost']:.7g}",
        "Reference / cost": f"{REFERENCE[name] / result['score']:.3f}",
    })
display(pd.DataFrame(rows).style.hide(axis="index"))
print(f"Full verification completed in {(time.monotonic() - started) / 60:.1f} min")
print(f"Verified local score: {SCORE:.5f}")
print(f"Worst reviewed-salt score: {WORST_SALT_SCORE:.5f}")


In [ ]:
MAX_ID = 2**30 // 1536
problems = []
all_files = sorted(path.relative_to(SUB).as_posix() for path in SUB.rglob("*") if path.is_file())
angles = [name for name in all_files if name.startswith("dfthcsos/angle_k_") and name.endswith(".kmb")]
expected = {"invsqrt.kmb", "dqi.kmb", "dfthcsos/submission.npz"} | {
    f"dfthcsos/angle_k_{index}.kmb" for index in range(len(angles))
}
if set(all_files) != expected:
    problems.append(
        f"manifest differs: extra={sorted(set(all_files) - expected)}, "
        f"missing={sorted(expected - set(all_files))}"
    )
for name in sorted(item for item in all_files if item.endswith(".kmb")):
    circuit = load_circuit(SUB / name)
    top_id = max(circuit.num_qubits, circuit.num_bits) - 1
    unnamed = [register for register in circuit.register_data if not isinstance(register, str)]
    if top_id >= MAX_ID:
        problems.append(f"{name}: largest ID {top_id} >= {MAX_ID}")
    if unnamed:
        problems.append(f"{name}: unnamed registers {unnamed}")
with np.load(SUB / "dfthcsos" / "submission.npz", allow_pickle=False) as archive:
    for name in archive.files:
        value = archive[name]
        expected_kind = np.integer if name in {"tree_edges", "angle_words"} else np.floating
        little = value.dtype.byteorder in {"<", "|"} or (
            value.dtype.byteorder == "=" and sys.byteorder == "little"
        )
        if not np.issubdtype(value.dtype, expected_kind) or not little:
            problems.append(f"submission.npz {name}: invalid dtype {value.dtype.str}")
if problems:
    raise AssertionError("\n".join(problems))

ZIP_PATH = WORK / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(SUB.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(SUB).as_posix())
with zipfile.ZipFile(ZIP_PATH) as archive:
    infos = archive.infolist()
    if len(infos) != len(expected) or {info.filename for info in infos} != expected:
        raise AssertionError("ZIP manifest differs from the verified submission tree")
    if any(info.file_size == 0 for info in infos):
        raise AssertionError("ZIP contains an empty member")
    if archive.testzip() is not None:
        raise AssertionError("ZIP CRC validation failed")
    member_hashes = {
        info.filename: hashlib.sha256(archive.read(info.filename)).hexdigest()
        for info in infos
    }

summary = {
    "score": SCORE,
    "worst_reviewed_salt_score": WORST_SALT_SCORE,
    "results": RESULTS,
    "dqi_salt_trials": dqi_trials,
    "source_plan": PLAN,
    "offset_refit": OFFSET_REFIT_REPORT,
    "member_sha256": member_hashes,
    "archive_sha256": sha256_file(ZIP_PATH),
    "archive_bytes": ZIP_PATH.stat().st_size,
}
(WORK / "summary_v6.json").write_text(json.dumps(summary, indent=2, default=str) + "\n")
(WORK / "submission-members-v6.sha256").write_text(
    "".join(f"{member_hashes[name]}  {name}\n" for name in sorted(member_hashes))
)
print(f"Submission ready: {ZIP_PATH}")
print(f"Files           : {len(expected)}")
print(f"Archive size    : {ZIP_PATH.stat().st_size / 2**20:.2f} MiB")
print(f"SHA-256         : {summary['archive_sha256']}")
print(f"Verified score  : {SCORE:.5f}")
print(f"Worst salt score: {WORST_SALT_SCORE:.5f}")
ZIP_PATH


## What this revision can and cannot change

`H2_I` is read directly from `submission.npz`; changing it does not require
recompiling the fixed ANGLE circuits. This makes the offset refit a genuine,
reviewable numerical optimization rather than a projected circuit count. The
full verifier remains the final authority.

The DQI and invsqrt artifacts are compiled binaries. Their missing EXP-119a
and EXP-225a generators prevent safe gate-level changes here. The notebook
therefore preserves their verified bytes. The next large revision needs either
the paired J-fraction generator for a <=900k-Toffoli DQI build or starter-kit
training that produces a quantized `(R,B,C)` checkpoint across the inner-QROAM
block-16 transition.
